# TF-IDF

#### TF-IDF singkatan dari Term Frequency – Inverse Document Frequency, yaitu metode statistical weighting yang dipakai di text mining untuk mengukur seberapa penting sebuah kata dalam sebuah dokumen dibandingkan dengan kumpulan dokumen lainnya (corpus).

In [1]:
!pip install plotly
!pip install --upgrade gensim


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
from gensim.models import Word2Vec, FastText
import pandas as pd
import re

from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

from matplotlib import pyplot as plt
import plotly.graph_objects as go

import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [4]:
pta_df_wfq = pd.read_csv("pta_word_frequency.csv")
print(pta_df_wfq.head(10))

         kata  jumlah
0    pengaruh    5551
1       kerja    5393
2      teliti    4381
3    variabel    3668
4       usaha    2535
5  signifikan    2489
6         uji    2364
7    karyawan    2265
8       nilai    1907
9       hasil    1784


In [5]:
df = pd.read_csv('pta_manajemen.csv')

In [8]:
print(df.columns)


Index(['id', 'penulis', 'judul', 'abstrak_id', 'abstrak_en',
       'pembimbing_pertama', 'pembimbing_kedua', 'prodi'],
      dtype='object')


## Cleaning Text

In [9]:
clean_txt = []

for text in df['abstrak_id']:
    desc = str(text).lower()                       # ubah ke huruf kecil
    desc = re.sub(r"<.*?>", " ", desc)             # hapus tag HTML
    desc = re.sub(r"[^a-zA-Z\s]", " ", desc)       # hapus angka & karakter non-huruf
    desc = re.sub(r"\s+", " ", desc).strip()       # hapus spasi berlebih
    clean_txt.append(desc)

df['clean'] = clean_txt
print(df[['abstrak_id', 'clean']].head())


                                          abstrak_id  \
0  ABSTRAK\r\nSatiyah, Pengaruh Faktor-faktor Pel...   
1  Tujuan penelitian ini adalah untuk mengetahui ...   
2                                                NaN   
3  Aplikasi nyata pemanfaatan teknologi informasi...   
4  Abstrak\r\nPenelitian ini menggunakan metode k...   

                                               clean  
0  abstrak satiyah pengaruh faktor faktor pelatih...  
1  tujuan penelitian ini adalah untuk mengetahui ...  
2                                                nan  
3  aplikasi nyata pemanfaatan teknologi informasi...  
4  abstrak penelitian ini menggunakan metode kuan...  


In [10]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['clean'])

# ubah ke DataFrame biar keliatan
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)

In [11]:
print("\nTF-IDF shape:", tfidf_df.shape)
print(tfidf_df.head())


TF-IDF shape: (1031, 7406)
   aaa  aaaamanahsyariah  aar  abadi  abalisis  abc  abcs  abd  abdul  \
0  0.0               0.0  0.0    0.0       0.0  0.0   0.0  0.0    0.0   
1  0.0               0.0  0.0    0.0       0.0  0.0   0.0  0.0    0.0   
2  0.0               0.0  0.0    0.0       0.0  0.0   0.0  0.0    0.0   
3  0.0               0.0  0.0    0.0       0.0  0.0   0.0  0.0    0.0   
4  0.0               0.0  0.0    0.0       0.0  0.0   0.0  0.0    0.0   

   abdullah  ...  zscore  zte  zuhri  zuhruf  zulfi  zulia  zuliana  zulkifli  \
0       0.0  ...     0.0  0.0    0.0     0.0    0.0    0.0      0.0       0.0   
1       0.0  ...     0.0  0.0    0.0     0.0    0.0    0.0      0.0       0.0   
2       0.0  ...     0.0  0.0    0.0     0.0    0.0    0.0      0.0       0.0   
3       0.0  ...     0.0  0.0    0.0     0.0    0.0    0.0      0.0       0.0   
4       0.0  ...     0.0  0.0    0.0     0.0    0.0    0.0      0.0       0.0   

   zulpah  zyn  
0     0.0  0.0  
1     0.0  0

## Word Embedding

#### Word Embedding adalah teknik representasi kata ke dalam bentuk vektor numerik (angka-angka) sehingga komputer bisa memahami hubungan antar kata dalam ruang multidimensi.

In [13]:
corpus = []
for col in df['clean']:
    word_list = col.split(" ")
    corpus.append(word_list)

# Word2Vec dengan ukuran vektor 50
model = Word2Vec(corpus, min_count=1, vector_size=50, window=5, sg=0)


In [14]:
print("Kata mirip dengan 'penelitian':")
print(model.wv.most_similar('penelitian', topn=5))

print("\nKata mirip dengan 'data':")
print(model.wv.most_similar('data', topn=5))

# contoh cosmul
print("\nCosmul (penelitian + sistem - data):")
print(model.wv.most_similar_cosmul(positive=['penelitian', 'sistem'], negative=['data'], topn=5))

# doesnt_match: cari kata yang tidak sesuai konteks
print("\nKata yang tidak cocok dalam ['penelitian', 'data', 'sistem', 'informasi']:")
print(model.wv.doesnt_match("penelitian data sistem informasi".split()))

# save embeddings
filename = 'pta_embeddings.txt'
model.wv.save_word2vec_format(filename, binary=False)
print(f"\nEmbeddings disimpan ke {filename}")

Kata mirip dengan 'penelitian':
[('adalah', 0.7808646559715271), ('peneitian', 0.7744374871253967), ('diskriptif', 0.7578980326652527), ('peneltian', 0.7534824013710022), ('semua', 0.7413105964660645)]

Kata mirip dengan 'data':
[('primer', 0.9515121579170227), ('sekunder', 0.9267135262489319), ('reduksi', 0.9266449809074402), ('teknik', 0.9162290096282959), ('angket', 0.9155388474464417)]

Cosmul (penelitian + sistem - data):
[('enrichmentsecara', 1.1692125797271729), ('kepemimpinan', 1.1315382719039917), ('menganalis', 1.1268906593322754), ('enlargementdan', 1.120581865310669), ('berjudul', 1.1128382682800293)]

Kata yang tidak cocok dalam ['penelitian', 'data', 'sistem', 'informasi']:
penelitian

Embeddings disimpan ke pta_embeddings.txt


In [15]:
class MyTokenizer:
    def fit_transform(self, texts):
        # Tokenisasi sederhana: lowercase + split
        return [str(text).lower().split() for text in texts]

class MeanEmbeddingVectorizer:
    def __init__(self, word2vec_model):
        self.word2vec = word2vec_model
        # Perbaikan: gunakan vector_size (Gensim ≥ 4.0)
        self.dim = word2vec_model.wv.vector_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_tokenized = MyTokenizer().fit_transform(X)
        embeddings = []
        for words in X_tokenized:
            # Ambil vektor hanya untuk kata yang ada di vocab
            valid_vectors = [
                self.word2vec.wv[word] for word in words
                if word in self.word2vec.wv
            ]
            if valid_vectors:
                embeddings.append(np.mean(valid_vectors, axis=0))
            else:
                embeddings.append(np.zeros(self.dim))
        return np.array(embeddings)

    def fit_transform(self, X, y=None):
        return self.transform(X)

In [16]:
df.shape

(1031, 9)

In [17]:
mean_embedding_vectorizer = MeanEmbeddingVectorizer(model)
mean_embedded = mean_embedding_vectorizer.fit_transform(df['clean'])

In [18]:
df['array']=list(mean_embedded)

In [21]:
df.head(10)

,id,penulis,judul,abstrak_id,abstrak_en,pembimbing_pertama,pembimbing_kedua,prodi,clean,array
0,80211100070,SATIYAH,PENGARUH FAKTOR-FAKTOR PELATIHAN DAN PENGEMBAN...,"ABSTRAK\r\nSatiyah, Pengaruh Faktor-faktor Pel...",ABSTRACT\r\n\r\nIn an effort to increase labor...,"Dra. Hj. S. Anugrahini Irawati, MM","Helmi Buyung Aulia,S,ST.SE,M.MT",Manajemen,abstrak satiyah pengaruh faktor faktor pelatih...,"[-0.003373819, -0.1866982, -0.04397953, -0.468..."
1,90211200001,Faishal,ANALISIS PERSEPSI BRAND ASSOCIATION MENURUT PE...,Tujuan penelitian ini adalah untuk mengetahui ...,This study wanted to know the brand associatio...,Nurita Andriani,Yustina Chrismardani,Manajemen,tujuan penelitian ini adalah untuk mengetahui ...,"[-0.504735, 0.022537036, -0.4537263, -0.057546..."
2,80211100050,Wahyu Kurniawan,PENGARUH GAYA KEPEMIMPINAN DEMOKRATIK TERHADAP...,NaN,NaN,"Dr. Dra. Hj. Iriani Ismail, MM","Dra. Hj. S. Anugrahini Irawati, MM",Manajemen,nan,"[0.00952131, 0.011932555, -0.0011965537, -0.01..."
3,100211200002,Muhammad Zakaria Utomo,Pengukuran Website Quality Pada Situs Sistem A...,Aplikasi nyata pemanfaatan teknologi informasi...,Academic portal system in University of Trunoj...,"Dr. Ir. Nurita Andriani, MM","Nirma Kurriwati, SP, M.Si",Manajemen,aplikasi nyata pemanfaatan teknologi informasi...,"[-0.09714826, -0.103862226, -0.29716364, -0.15..."
4,80211100044,Hendri Wahyudi Prayitno,PENGARUH KEPEMIMPINAN DAN KOMPENSASI TERHADAP ...,Abstrak\r\nPenelitian ini menggunakan metode k...,Abstract\r\nThis research use quantitative met...,"Dra. Hj. S Anugrahini Irawati, MM","Helmi Buyung Aulia,S,ST,SE,.MT",Manajemen,abstrak penelitian ini menggunakan metode kuan...,"[0.06521979, -0.10184082, 0.20445642, -0.89799..."
5,80211100119,Aththaariq,PENGARUH KOMPETENSI DOSEN TERHADAP KINERJA DOS...,"Abstrak\r\n\r\nAththaariq, Pengaruh Kompetensi...",Abstract\r\n\r\nThis study is aimed to analyze...,"Dr.RM Moch Wispandono,.S.E,.MS","Dr. Muhammad Alkirom Wildan,S.E.,M.Si.",Manajemen,abstrak aththaariq pengaruh kompetensi dosen t...,"[-0.075939305, -0.1406312, -0.16101375, -0.461..."
6,80211100103,Haryono Arifin,PENGARUH PERILAKU KONSUMEN TERHADAP KEPUTUSAN ...,"ABSTRAK\r\nHaryono Arifin, Pengaruh Perilaku K...","ABSTRACT\r\n\r\nHaryono Arifin, Influence Cons...","Dr. Ir Nurita Andriani, MM","Nirma Kurriwati, S.P., M.Si",Manajemen,abstrak haryono arifin pengaruh perilaku konsu...,"[0.07858666, -0.2827331, -0.414383, -0.2376783..."
7,80211100098,Dharma Abidin Syah,PENGARUH TIPE KEPEMIMPINAN TERHADAP PRESTASI K...,"ABSTRAK\r\n\tDharma Abidin Syah,Kesimpulan: (1...","ABSTRACT\r\n\r\nDharma Abidin Syah,Conclusions...","Drs. Ec. Mudji Kuswinarno M,Si","Faidal SE, MM",Manajemen,abstrak dharma abidin syah kesimpulan terdapat...,"[0.32095918, -0.36276254, 0.33065572, -0.67049..."
8,90211100079,Toni Budianto,PENGARUH DIMENSI KUALITAS PELAYANAN TERHADAP K...,ABSTRAK\r\n\r\nTujuan penelitian ini adalah un...,ABSTRACT\r\n\r\nThe purpose of this study was ...,"Bambang Setiyo Pambudi, S.E., MM.","Fathor AS, S.E., MM.",Manajemen,abstrak tujuan penelitian ini adalah untuk men...,"[-0.14979018, -0.34985632, 0.048089612, -0.622..."
9,90211100089,Iwan Kurniawan Gomes,ANALISIS TINGKAT RISIKO KREDIT \r\nPADA PD. BP...,Hasil dari penelitian ini dari perhitungan Cre...,Results from this study of Credit Risk Ratio c...,"Drs. Ec. Makhmud Zulkifli, M.Si","Echsan Gani, S.E., M.SI",Manajemen,hasil dari penelitian ini dari perhitungan cre...,"[0.011993256, -0.007067294, -0.42419395, -0.31..."


In [22]:
df['embedding_length'] = df['array'].str.len()

In [23]:
print(df['embedding_length'])

0       50
1       50
2       50
3       50
4       50
        ..
1026    50
1027    50
1028    50
1029    50
1030    50
Name: embedding_length, Length: 1031, dtype: int64


In [24]:
df.shape

(1031, 11)

In [25]:
num_features = len(df['array'].iloc[0])  # asumsi semua list punya panjang sama
columns = [f'f{i+1}' for i in range(num_features)]

# Inisialisasi dictionary untuk menampung data per kolom
data_dict = {col: [] for col in columns}

# Looping setiap baris di kolom 'embedding'
for embedding_list in df['array']:
    for i, value in enumerate(embedding_list):
        data_dict[f'f{i+1}'].append(value)

# Buat DataFrame dari dictionary
embedding_df = pd.DataFrame(data_dict)

print(embedding_df)

            f1        f2        f3        f4        f5        f6        f7  \
0    -0.003374 -0.186698 -0.043980 -0.468721  0.092273 -0.625375  0.872841   
1    -0.504735  0.022537 -0.453726 -0.057547  0.198439 -0.308800  0.571099   
2     0.009521  0.011933 -0.001197 -0.014948  0.012792 -0.006262  0.005304   
3    -0.097148 -0.103862 -0.297164 -0.150468  0.189467 -0.247490  0.638271   
4     0.065220 -0.101841  0.204456 -0.897993  0.375457 -0.642278  0.928632   
...        ...       ...       ...       ...       ...       ...       ...   
1026 -0.066936 -0.015970 -0.373219 -0.150726 -0.138300 -0.416050  0.749273   
1027  0.182084 -0.410338  0.240756 -0.616924  0.012590 -0.760157  1.058336   
1028 -0.157459 -0.012404 -0.103568 -0.430279  0.201682 -0.418780  0.725307   
1029  0.020639 -0.352344 -0.379155 -0.299957  0.507706 -0.392144  0.477148   
1030  0.094403 -0.046604 -0.060677 -0.652003  0.230387 -0.612119  1.003906   

            f8        f9       f10  ...       f41       f42    

In [26]:
embedding_df

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f41,f42,f43,f44,f45,f46,f47,f48,f49,f50
0,-0.003374,-0.186698,-0.043980,-0.468721,0.092273,-0.625375,0.872841,0.533811,0.207106,-0.419236,...,0.357293,-0.365203,0.161903,-0.225028,0.913188,-0.249704,0.546130,-0.716899,0.555572,1.131540
1,-0.504735,0.022537,-0.453726,-0.057547,0.198439,-0.308800,0.571099,0.032137,0.193959,-0.358577,...,0.524794,-0.614485,0.503177,-0.238595,0.247937,-0.132879,0.201781,-0.760950,0.363967,1.066985
2,0.009521,0.011933,-0.001197,-0.014948,0.012792,-0.006262,0.005304,-0.018428,-0.017723,-0.007998,...,-0.016504,0.003701,0.006011,-0.011140,0.018107,0.013669,0.005570,0.008566,0.008986,0.013651
3,-0.097148,-0.103862,-0.297164,-0.150468,0.189467,-0.247490,0.638271,0.137313,0.141832,-0.415341,...,0.429197,-0.476581,0.288238,-0.131972,0.487580,-0.206289,0.164133,-0.542897,0.459323,0.926310
4,0.065220,-0.101841,0.204456,-0.897993,0.375457,-0.642278,0.928632,0.594240,0.400300,-0.503492,...,0.665431,-0.779258,0.503562,-0.207495,0.914278,-0.510216,0.736533,-1.166116,0.504775,1.544114
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1026,-0.066936,-0.015970,-0.373219,-0.150726,-0.138300,-0.416050,0.749273,0.523464,-0.221615,-0.635367,...,0.509503,-0.225866,0.514827,-0.029728,0.452368,-0.413775,0.280740,-0.682866,0.243510,0.946561
1027,0.182084,-0.410338,0.240756,-0.616924,0.012590,-0.760157,1.058336,1.008561,0.062664,-1.158819,...,0.470729,-0.323484,0.201670,0.043589,1.291648,-0.672726,0.661616,-1.071737,0.876984,1.359840
1028,-0.157459,-0.012404,-0.103568,-0.430279,0.201682,-0.418780,0.725307,0.466418,0.234952,-0.554781,...,0.526190,-0.600188,0.657847,-0.163055,0.519988,-0.373793,0.430978,-1.077568,0.509147,1.417799
1029,0.020639,-0.352344,-0.379155,-0.299957,0.507706,-0.392144,0.477148,0.074897,0.463377,-0.372824,...,0.584020,-1.041220,0.219735,-0.329617,0.742128,-0.148023,0.135037,-0.867374,0.447756,1.330320


In [28]:
embedding_df.shape

(1031, 50)